# Fase 2 · herramientas
ReAct clásico queda aislado. AutoGen organiza conversaciones entre agentes; aquí el integrador usa un agente con herramientas verificables.

In [1]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

In [2]:
from pedro.catalog import crear_escenario
from pedro.legacy import RouterCompras, ejecutar_ruta
from pedro.chains import composicion_paralela
from pedro.integrations import consultar_cep
escenario = crear_escenario(Path('storage/notebooks/catalogos'))
router = RouterCompras()
print(ejecutar_ruta(router, escenario, 'precio de PEC-01-0001'))
print(ejecutar_ruta(router, escenario, 'PEC-01-0001'))
print(composicion_paralela(escenario).invoke({'producto':'Compresor'}))
print(consultar_cep('01001000', simulado=True))

{'ruta': 'comparacion', 'resultado': [{'dataset_id': 'df7b789b-83d1-4edc-9b48-b643293740db', 'id': 'PEC-01-0001-BR-05', 'producto_id': 'PEC-01-0001', 'proveedor_id': 'BR-05', 'proveedor_nombre': 'Litoral Parts', 'precio_unitario_centavos': 37020, 'flete_centavos': 7940, 'stock': 0, 'plazo_dias': 12, 'pedido_minimo': 5, 'nacional': True, 'meses_historial': 5, 'precio_anterior_centavos': 42977, 'precio_medio_centavos': 41786, 'precio_minimo_centavos': 37020, 'precio_maximo_centavos': 45956}, {'dataset_id': 'df7b789b-83d1-4edc-9b48-b643293740db', 'id': 'PEC-01-0001-BR-06', 'producto_id': 'PEC-01-0001', 'proveedor_id': 'BR-06', 'proveedor_nombre': 'CentroPeças', 'precio_unitario_centavos': 143892, 'flete_centavos': 8523, 'stock': 67, 'plazo_dias': 2, 'pedido_minimo': 5, 'nacional': True, 'meses_historial': 5, 'precio_anterior_centavos': 118987, 'precio_medio_centavos': 137251, 'precio_minimo_centavos': 118987, 'precio_maximo_centavos': 159111}, {'dataset_id': 'df7b789b-83d1-4edc-9b48-b6432

## R09 · routing dinámico y composición
La primera celda de este notebook ejecuta `RouterCompras` (una `RouterChain` con reglas), `ejecutar_ruta` (cadena secuencial: decidir la ruta y ejecutar su cadena) y `composicion_paralela` (`RunnableParallel`). Aquí se añade el router decidido por un modelo (`LLMRouterChain`). **Doble de prueba:** `FakeListLLM` devuelve el destino ya escrito; la comparación con el modelo real está en `docs/trazas/r09_router_llm.json`.

In [3]:
from langchain_core.language_models.fake import FakeListLLM
from pedro.legacy import DESTINOS, crear_router_llm
print('destinos:', DESTINOS)
for pregunta, destino in (('¿Cuánto cuesta el compresor PEC-01-0001?', 'comparacion'),
                          ('¿A qué ciudad corresponde el CEP 01001-000?', 'cep'),
                          ('Hola, buenos días', 'DEFAULT')):
    doble = FakeListLLM(responses=['```json\n{"destination": "' + destino + '", "next_inputs": "x"}\n```'])
    salida = ejecutar_ruta(crear_router_llm(doble), escenario, pregunta)
    print(pregunta, '->', salida['ruta'], '|', str(salida['resultado'])[:110])
    print('   el router por reglas habría elegido:', ejecutar_ruta(router, escenario, pregunta)['ruta'])

destinos: {'catalogo': 'buscar o describir una pieza por nombre, SKU o ID, sin hablar de precios', 'comparacion': 'precios, costes, ofertas, proveedores, stock o plazos de una pieza', 'cep': 'confirmar la ciudad o el estado de un código postal brasileño (CEP)'}
¿Cuánto cuesta el compresor PEC-01-0001? -> comparacion | [{'dataset_id': 'df7b789b-83d1-4edc-9b48-b643293740db', 'id': 'PEC-01-0001-BR-05', 'producto_id': 'PEC-01-0001
   el router por reglas habría elegido: catalogo
¿A qué ciudad corresponde el CEP 01001-000? -> cep | {'estado': 'ok', 'cep': '01001000', 'cidade': 'Cidade Simulada', 'uf': 'SP', 'modo': 'simulado'}
   el router por reglas habría elegido: catalogo
Hola, buenos días -> aclaracion | ¿Buscas una pieza, sus ofertas o un CEP?
   el router por reglas habría elegido: catalogo


## R08 · agente ReAct con initialize_agent, herramientas y memoria
**Doble de prueba:** `FakeListLLM` devuelve un guion fijo de pensamientos y acciones; no decide nada. Lo real es el agente (`initialize_agent` con `ZERO_SHOT_REACT_DESCRIPTION`), las dos herramientas ejecutadas sobre el catálogo y `ConversationBufferMemory`. La misma conversación con el modelo real, donde el LLM sí elige, está en `docs/trazas/r08_react_llm.json`.

In [4]:
from langchain_core.language_models.fake import FakeListLLM
from pedro.legacy import crear_react_textual
llm = FakeListLLM(responses=[
    'Thought: debo buscar la pieza\nAction: buscar_pieza\nAction Input: NEVE-COMP-0001',
    'Thought: ya la tengo\nFinal Answer: el ID es PEC-01-0001',
    'Thought: el historial dice PEC-01-0001\nAction: ofertas_de_pieza\nAction Input: PEC-01-0001',
    'Thought: comparo los plazos\nFinal Answer: CentroPeças entrega en 2 días'])
react, buffer = crear_react_textual(llm, escenario)
print('variables del prompt:', sorted(react.agent.llm_chain.prompt.input_variables))
for pregunta in ('Busca el compresor con SKU NEVE-COMP-0001 y dime su ID.',
                 '¿Cuál de sus proveedores entrega más rápido?'):
    salida = react.invoke({'input': pregunta})
    for accion, observacion in salida['intermediate_steps']:
        print('  acción:', accion.tool, '| entrada:', accion.tool_input, '| observación:', str(observacion)[:150])
    print('  respuesta:', salida['output'])
print('memoria de buffer:')
print(buffer.load_memory_variables({})['chat_history'])

variables del prompt: ['agent_scratchpad', 'chat_history', 'input']
  acción: buscar_pieza | entrada: NEVE-COMP-0001 | observación: [{'id': 'PEC-01-0001', 'sku': 'NEVE-COMP-0001', 'name': 'Compresor para Neveras NEVE-COMP-0001', 'category': 'Neveras'}]
  respuesta: el ID es PEC-01-0001
  acción: ofertas_de_pieza | entrada: PEC-01-0001 | observación: [{'proveedor': 'Litoral Parts', 'precio_unitario_centavos': 37020, 'stock': 0, 'plazo_dias': 12}, {'proveedor': 'CentroPeças', 'precio_unitario_centav
  respuesta: CentroPeças entrega en 2 días
memoria de buffer:
Human: Busca el compresor con SKU NEVE-COMP-0001 y dime su ID.
AI: el ID es PEC-01-0001
Human: ¿Cuál de sus proveedores entrega más rápido?
AI: CentroPeças entrega en 2 días


/home/efraxpc/infnet/agentes/pedro_asistente_de_compras/src/pedro/legacy.py:77: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memoria = ConversationBufferMemory(memory_key="chat_history", input_key="input", output_key="output")
/home/efraxpc/infnet/agentes/pedro_asistente_de_compras/.venv/lib/python3.13/site-packages/pydantic/main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated

La llamada real a ViaCEP se registra por separado en docs/evidencias.md.

## R12 · bitácora de un turno con error y recuperación
Los cuatro componentes de confiabilidad en un solo turno del flujo final. **Doble de prueba:** el modelo es un guion; ViaCEP está simulado. La primera propuesta del guion es texto que no es JSON (error del parser); la segunda es válida. El callback `Traza` registra cada llamada a herramienta y cada verificación, y `RunnableWithMessageHistory` guarda en memoria solo la pregunta y la respuesta publicada.

In [5]:
import json
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda
from pedro.agent import Traza, crear_flujo
from pedro.memory import Historial, nueva_sesion
memoria = Path('storage/notebooks/r12.sqlite3')
sesion = nueva_sesion(memoria, 'alumna', escenario.dataset_id)
llamar = AIMessage(content='', tool_calls=[{'name': 'consultar_cep', 'args': {'cep': '01001000'}, 'id': 'c1'}])
valida = json.dumps({'estado': 'cep', 'dataset_id': escenario.dataset_id, 'solicitud': {'cep': '01001000'}})
guion = iter([llamar, AIMessage(content='La ciudad es São Paulo, creo.'), llamar, AIMessage(content=valida)])
traza = Traza()
flujo = crear_flujo(escenario, memoria, llm=RunnableLambda(lambda prompt, **kw: next(guion)),
                    cep_simulado=True, traza=traza)
respuesta = flujo.invoke({'input': '¿A qué ciudad corresponde el CEP 01001000?'},
                         config={'configurable': {'user_id': 'alumna', 'session_id': sesion}})
for evento in traza.eventos:
    print({k: (str(v)[:70] if isinstance(v, str) else v) for k, v in evento.items()})
print('publicado:', respuesta.content)
print('memoria:', [(m.type, m.content[:60]) for m in Historial(memoria, 'alumna', sesion).todos()])

{'evento': 'tool_start', 'ruta': 'consultar_cep', 'herramienta': 'consultar_cep', 'entrada': "{'cep': '01001000'}"}
{'evento': 'tool_end', 'resultado': '{"estado": "ok", "cep": "01001000", "cidade": "Cidade Simulada", "uf":', 'fuentes': []}
{'evento': 'verificacion', 'intento': 1, 'estado': None, 'errores': ['OutputParserException']}
{'evento': 'tool_start', 'ruta': 'consultar_cep', 'herramienta': 'consultar_cep', 'entrada': "{'cep': '01001000'}"}
{'evento': 'tool_end', 'resultado': '{"estado": "ok", "cep": "01001000", "cidade": "Cidade Simulada", "uf":', 'fuentes': []}
{'evento': 'cep_anclado', 'estado': 'anclada', 'cep': '01001000'}
{'evento': 'verificacion', 'intento': 2, 'estado': 'cep', 'errores': []}
publicado: CEP 01001000: Cidade Simulada/SP (ViaCEP, modo simulado).
memoria: [('human', '¿A qué ciudad corresponde el CEP 01001000?'), ('ai', 'CEP 01001000: Cidade Simulada/SP (ViaCEP, modo simulado).')]


/tmp/ipykernel_52969/467001735.py:12: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  flujo = crear_flujo(escenario, memoria, llm=RunnableLambda(lambda prompt, **kw: next(guion)),


Lectura: la bitácora muestra la herramienta llamada dos veces, el primer intento rechazado con `OutputParserException`, el segundo aceptado y el CEP rellenado desde la observación. La propuesta rechazada no llega a la memoria: solo se guardan la pregunta y la respuesta publicada.

## R07 · qué aporta el orquestador: inventario de piezas de LangChain en Pedro
Sin modelo. El inventario se obtiene leyendo los imports del código (`pedro.anatomia.inventario_langchain`), no se escribe a mano. Sirve para argumentar la elección con hechos: cada fila es una pieza que el proyecto no tuvo que construir. La comparación con otros orquestadores está en `docs/respuestas_rubricas.md` §R07 y es conceptual: aquí solo se ejecutó LangChain.

In [6]:
from pedro.anatomia import inventario_langchain
inventario = inventario_langchain()
for pieza, dato in inventario.items():
    print(f"{pieza:38} {', '.join(dato['nombres'])}")
    print(f"{'':38} usado en: {', '.join(dato['archivos'])}")
print('piezas:', len(inventario), '| nombres importados:', sum(len(d['nombres']) for d in inventario.values()))

agentes y ejecutor                     AgentExecutor, AgentType, create_openai_tools_agent, initialize_agent
                                       usado en: agent.py, legacy.py, orquestadores.py
almacén vectorial (FAISS)              FAISS
                                       usado en: policy_demo.py, rag.py
cadenas clásicas (RouterChain)         ConversationChain, LLMRouterChain, MULTI_PROMPT_ROUTER_TEMPLATE, RouterChain, RouterOutputParser
                                       usado en: legacy.py
callbacks y trazas                     BaseCallbackHandler
                                       usado en: agent.py, laboratorio.py
composición (Runnable, LCEL, @chain)   ConfigurableFieldSpec, RunnableConfig, RunnableLambda, RunnableParallel, RunnablePassthrough, RunnableWithMessageHistory, chain
                                       usado en: agent.py, anatomia.py, chains.py, laboratorio.py, orquestadores.py, precios.py
documentos y embeddings                Document, Embeddings, Fas